Исторический исследовательский ноутбук. Сохранён как запись этапов работы; выводы очищены, код не изменён. Ячейки содержат альтернативные сценарии и старые пути. Для повторения актуальных расчётов используйте `dissertation/run_analysis.py` и инструкции в корневом README.


## Влияния зависимости PARL (представленность женщин в парламенте) на всякое

### 0. Библиотеки

In [ ]:
import pandas as pd
import numpy as np

import statsmodels.api as sm
import statsmodels.formula.api as smf

import random

import plotly.graph_objects as go
import plotly.express as px

In [ ]:
import warnings
warnings.filterwarnings("ignore")

### 1. Данные

In [ ]:
file_path = '../data/final_data/final_data.csv'

df = pd.read_csv(file_path, sep = ';', decimal = ',')
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
# кол-во пропусков
print(df.isnull().sum())

### 2. График

In [ ]:
selected_countries = [
    'Nicaragua',
    'United Arab Emirates',
    'Mexico',
    'Moldova'
]

legend_order = (
    df[
        df['Year'].between(2018, 2022) &
        df['Country Name'].isin(selected_countries)
    ]
    .groupby('Country Name')['PARL']
    .mean()
    .sort_values(ascending=False)
)

df_filtered = df[['Country Name', 'Year', 'PARL']].dropna().drop_duplicates()
random_countries = random.sample(legend_order.index.tolist(), k=4)

fig = go.Figure()

for country in legend_order.index:
    country_data = df_filtered[df_filtered['Country Name'] == country]
    fig.add_trace(go.Scatter(
        x=country_data['Year'],
        y=country_data['PARL'],
        mode='lines',
        name=country,
        visible=True if country in random_countries else 'legendonly'
    ))

fig.update_layout(
    template='plotly_white',
    title='Динамика PARL по годам',
    height=750,
    legend_title_text='Страна'
)

fig.show()
fig.write_image("graphs/graphs/PARL.png", width=1600, height=900, scale=4)

### 3. Кластеризация

In [ ]:
# Средний PARL по стране за 2018–2022
parl_by_country = df.groupby('Country Name', as_index=False)['PARL'].mean()

# Делим на три группы по перцентилям
low = parl_by_country['PARL'].quantile(0.33)
high = parl_by_country['PARL'].quantile(0.66)

def assign_group(parl):
    if parl <= low:
        return 'Low'
    elif parl <= high:
        return 'Medium'
    else:
        return 'High'

parl_by_country['parl_group'] = parl_by_country['PARL'].apply(assign_group)

# Присоединяем обратно
df = df.merge(parl_by_country[['Country Name', 'parl_group']], on='Country Name', how='left')

In [ ]:
# Задаём нужный порядок
desired_order = ['Low', 'Medium', 'High']
df['parl_group'] = pd.Categorical(df['parl_group'],
                                           categories=desired_order,
                                           ordered=True)

# Теперь график будет от Q4 к Q1
fig = px.box(df,
             x='parl_group',
             y='Corruption',
             color='parl_group',
             category_orders={'parl_group': desired_order},
             title='Коррупция по странам',
             template='plotly_white',
             labels={'parl_group': 'PARL', 'Corruption': 'Индекс коррупции'})
fig.show()
fig.write_image("graphs/graphs/PARL_corruption.png", width=1600, height=900, scale=4)

### 4. Модель

In [ ]:
# Берем только последние 5 лет
df_recent = df[df['Year'].between(2018, 2022)].copy()

# Убираем строки с пропущенными значениями
df_recent = df_recent.dropna(subset=['PARL', 'Corruption', 'parl_group'])

# Убедимся, что parl_group — категориальный с нужным порядком
df_recent['parl_group'] = pd.Categorical(df_recent['parl_group'],
                                         categories=['Low', 'Medium', 'High'],
                                         ordered=True)


In [ ]:
models = {}

for cluster in df_recent['parl_group'].cat.categories:
    data = df_recent[df_recent['parl_group'] == cluster]

    X = data[['PARL']]
    X = sm.add_constant(X)
    y = data['Corruption']

    model = sm.OLS(y, X).fit()
    models[cluster] = model

    print(f"\n=== Модель для кластера '{cluster}' ===")
    print(model.summary())

In [ ]:
# Добавим логарифм GDP
df_panel = df[df['Year'].between(2018, 2022)].copy()
df_panel = df_panel[df_panel['GDP'] > 0].dropna(subset=[
    'Corruption', 'PARL', 'GDP', 'Age Dependency',
    'Rule of Law', 'Voice and Accountability', 'parl_group'
])
df_panel['log_GDP'] = np.log(df_panel['GDP'])


In [ ]:
models = {}

for cluster in df_panel['parl_group'].unique():
    data = df_panel[df_panel['parl_group'] == cluster]

    model = smf.ols(
        'Corruption ~ PARL + log_GDP + Q("Age Dependency") + Q("Voice and Accountability")',
        data=data
    ).fit(cov_type='cluster', cov_kwds={'groups': data['Country Name']})

    models[cluster] = model
    print(f"\n=== Модель с кластерными SE для кластера '{cluster}' ===")
    print(model.summary())
